In [1]:
import os
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:
TOKEN = "8187e2598869784bd77549217483e7b010ba949480497f1a43d30b30548fa697"

OUTPUT_DIR = "laji_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)

web_site_to_avoid = ["https://id.digitarium.fi/"]
web_site_to_avoid = []

API_URL = "https://api.laji.fi/v0/warehouse/query/unit/list"

params = {
    "collectionId": "HR.42",
    "informalTaxonGroupId": "MVL.232",
    "informalTaxonGroupIdNot": "MVL.1004,MVL.38,MVL.236,MVL.235,MVL.234",
    "selected": "unit.media",
    "pageSize": 100,
    "page": 1,
    "access_token": TOKEN
}

# session avec retry automatique
session = requests.Session()

retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)

session.mount("https://", HTTPAdapter(max_retries=retries))

In [3]:
r = session.get(API_URL, params=params, timeout=30)
r.raise_for_status()

data = r.json()

total_results = data["total"]
page_size = params["pageSize"]

max_pages = (total_results // page_size) + 1

print("Total observations:", total_results)
print("Max pages:", max_pages)

Total observations: 1385637
Max pages: 13857


In [6]:
import logging
import sys

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Handler fichier
file_handler = logging.FileHandler("download.log")
file_handler.setLevel(logging.INFO)

# Handler stdout
stream_handler = logging.StreamHandler(stream=sys.stdout)
stream_handler.setLevel(logging.INFO)

logger.addHandler(file_handler)
logger.addHandler(stream_handler)

def download(url):

    name = url.split("/")[-1]
    path = os.path.join(OUTPUT_DIR, name)

    if os.path.exists(path):
        return

    try:
        r = session.get(url, stream=True, timeout=(10, 15))  
        # (connexion max 10s, téléchargement max 15s)

        if r.status_code != 200:
            logger.info("skip %s status code %d", url, r.status_code)
            return

        with open(path, "wb") as f:
            for chunk in r.iter_content(8192):
                logger.info("downloading %s", url)
                f.write(chunk)
        
        return (url, path)

    except requests.exceptions.Timeout:
        logger.info("skip timeout %s", url)

    except requests.exceptions.RequestException:
        logger.info("skip error %s", url)

def check_web_site(url):
    if not url :
        return False
    
    for site in web_site_to_avoid:
        if url.startswith(site):
            return False
    return True

In [8]:
import tqdm
import json

dict_url_to_img = {}

num = 0
for page in (pbar := tqdm.tqdm(range(1, max_pages + 1))):

    params["page"] = page

    r = session.get(API_URL, params=params, timeout=30)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])

    for obs in results:
        media = obs.get("unit", {}).get("media", [])
        for m in media:
            url = m.get("fullURL") or m.get("largeURL") or m.get("thumbnailURL")
            if check_web_site(url):
                res = download(url)
                if res is not None:
                    num += 1
                    dict_url_to_img[res[0]] = res[1]
   
    pbar.set_description(f"Downloaded {num} images")

    json.dump(dict_url_to_img, open("url_to_img.json", "w"), indent=2)



Downloaded 0 images:   0%|          | 6/13857 [00:25<16:15:25,  4.23s/it]


KeyboardInterrupt: 